# 09 - Advanced Retrieval Techniques 🚀

## Learning Objectives 🎯

In this notebook, you'll master advanced retrieval techniques to build production-ready RAG systems:

1. **MultiQueryRetriever** - Generate multiple query perspectives for better retrieval
2. **Contextual Compression** - Extract only relevant portions to reduce tokens and costs
3. **Custom Retrievers** - Build retrievers with custom logic and validation
4. **Advanced Combinations** - Combine multiple techniques for optimal results
5. **Benchmarking** - Compare techniques and measure performance
6. **Production Patterns** - Best practices for real-world applications

---

## Table of Contents 📚

1. [Introduction to Advanced Retrieval](#intro)
2. [Setup & Data Preparation](#setup)
3. [MultiQueryRetriever - Multiple Perspectives](#multiquery)
4. [Contextual Compression - Token Efficiency](#compression)
5. [Custom Retrievers - Full Control](#custom)
6. [Advanced Combinations](#combinations)
7. [Comparison & Benchmarking](#benchmarking)
8. [Best Practices & Production Tips](#best-practices)
9. [Summary & Exercises](#summary)

---

<a id='intro'></a>
## 1. Introduction to Advanced Retrieval 🔍

### Why Advanced Retrieval Techniques?

Basic similarity search has limitations:

❌ **Query Phrasing Problem**: Different phrasings of the same question retrieve different results
❌ **Irrelevant Content**: Retrieved documents contain unnecessary information
❌ **Token Waste**: Feeding entire documents to LLMs increases cost and latency
❌ **Generic Logic**: One-size-fits-all retrieval doesn't work for all use cases

### Advanced Techniques Solve These Problems:

| Technique | Problem It Solves | Benefit |
|-----------|------------------|----------|
| **MultiQueryRetriever** | Query phrasing variations | Better recall, diverse perspectives |
| **Contextual Compression** | Irrelevant content in documents | Reduced tokens, lower cost |
| **Custom Retrievers** | Business-specific logic | Full control, specialized behavior |

### When to Use Advanced Techniques:

✅ **MultiQueryRetriever** when:
- Users phrase questions differently
- You need comprehensive coverage
- Quality matters more than speed

✅ **Contextual Compression** when:
- Retrieved documents are long
- Token costs are high
- You need only relevant excerpts

✅ **Custom Retrievers** when:
- You have specific business logic (filtering, scoring)
- You need to combine multiple retrieval methods
- You want full control over retrieval behavior

---

<a id='setup'></a>
## 2. Setup & Data Preparation ⚙️

Let's set up our environment and create a document collection for all examples.

In [9]:
# Setup: Import required libraries
from dotenv import load_dotenv
import os
from pathlib import Path

# Load environment variables
load_dotenv()

# Core LangChain imports
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Advanced retriever imports
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter
from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun

# Utility imports
from typing import List
import logging

# Verify versions
import langchain
print(f"✅ LangChain version: {langchain.__version__}")
print("✅ Setup complete!")

✅ LangChain version: 0.3.27
✅ Setup complete!


### Create Sample Document Collection

We'll create a diverse document collection about AI and machine learning topics.

In [10]:
# Create sample documents about AI and Machine Learning
documents = [
    Document(
        page_content="""Machine Learning Fundamentals
        
        Machine learning is a subset of artificial intelligence that enables systems to learn and improve 
        from experience without being explicitly programmed. The core concept involves algorithms that can 
        learn patterns from data and make predictions or decisions based on that learning.
        
        There are three main types of machine learning: supervised learning (learning from labeled data), 
        unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning 
        through interaction with an environment).
        
        Common applications include image recognition, natural language processing, recommendation systems, 
        and autonomous vehicles. The field has seen exponential growth due to increases in computational 
        power and the availability of large datasets.""",
        metadata={"source": "ml_fundamentals.txt", "category": "basics", "topic": "machine_learning"}
    ),
    Document(
        page_content="""Deep Learning and Neural Networks
        
        Deep learning is a specialized branch of machine learning based on artificial neural networks. 
        These networks are inspired by the structure and function of the human brain, consisting of 
        interconnected layers of nodes (neurons) that process information.
        
        Deep neural networks contain multiple hidden layers between the input and output layers, allowing 
        them to learn hierarchical representations of data. Each layer learns to recognize increasingly 
        complex patterns - for example, in image recognition, early layers might detect edges, middle 
        layers detect shapes, and later layers detect complete objects.
        
        Popular architectures include Convolutional Neural Networks (CNNs) for image processing, Recurrent 
        Neural Networks (RNNs) for sequential data, and Transformers for natural language processing. 
        Training deep networks requires large amounts of data and computational resources.""",
        metadata={"source": "deep_learning.txt", "category": "advanced", "topic": "neural_networks"}
    ),
    Document(
        page_content="""Natural Language Processing (NLP)
        
        Natural Language Processing is a field at the intersection of computer science, artificial 
        intelligence, and linguistics. It focuses on enabling computers to understand, interpret, and 
        generate human language in a meaningful way.
        
        Key NLP tasks include text classification, named entity recognition, sentiment analysis, machine 
        translation, question answering, and text summarization. Modern NLP heavily relies on deep learning 
        techniques, particularly transformer-based models like BERT, GPT, and T5.
        
        The transformer architecture, introduced in the 'Attention is All You Need' paper, revolutionized 
        NLP by using self-attention mechanisms to process sequential data in parallel. This led to 
        significant improvements in translation quality, text generation, and language understanding.
        
        Recent advances include large language models (LLMs) trained on massive text corpora, capable of 
        few-shot and zero-shot learning across diverse tasks.""",
        metadata={"source": "nlp_overview.txt", "category": "advanced", "topic": "nlp"}
    ),
    Document(
        page_content="""Retrieval-Augmented Generation (RAG)
        
        RAG is a powerful technique that combines the strengths of retrieval systems and generative 
        language models. Instead of relying solely on the knowledge encoded in a language model's parameters, 
        RAG retrieves relevant documents from an external knowledge base and uses them as context for 
        generation.
        
        The RAG process involves three main steps: (1) Query formulation and retrieval - converting user 
        queries into embeddings and retrieving similar documents from a vector database, (2) Context 
        augmentation - combining retrieved documents with the user query, and (3) Response generation - 
        using an LLM to generate answers based on the augmented context.
        
        Benefits of RAG include: reduced hallucination (answers are grounded in retrieved facts), ability 
        to update knowledge without retraining (just update the knowledge base), source attribution 
        (can cite where information came from), and cost-effectiveness (smaller LLMs can be used).
        
        RAG is particularly useful for question answering systems, chatbots, and applications requiring 
        up-to-date or specialized knowledge.""",
        metadata={"source": "rag_explained.txt", "category": "advanced", "topic": "rag"}
    ),
    Document(
        page_content="""Vector Embeddings and Similarity Search
        
        Vector embeddings are dense numerical representations of data (text, images, audio) in a 
        high-dimensional space. In NLP, word embeddings capture semantic relationships - words with 
        similar meanings are positioned closer together in the vector space.
        
        Modern embedding models like OpenAI's text-embedding-ada-002 or text-embedding-3-small can 
        encode entire sentences or documents into fixed-length vectors. These embeddings capture semantic 
        meaning, allowing for similarity comparisons using distance metrics like cosine similarity or 
        Euclidean distance.
        
        Vector databases (like FAISS, Pinecone, Weaviate, or Chroma) are optimized for storing and 
        searching large collections of embeddings. They use techniques like approximate nearest neighbor 
        (ANN) search to quickly find similar vectors even in billions of data points.
        
        Similarity search is the foundation of modern retrieval systems, enabling semantic search that 
        goes beyond keyword matching to understand the meaning and intent behind queries.""",
        metadata={"source": "embeddings.txt", "category": "intermediate", "topic": "embeddings"}
    ),
    Document(
        page_content="""Transformer Architecture
        
        The Transformer architecture, introduced by Vaswani et al. in 2017, has become the foundation 
        of modern NLP. Unlike previous sequential models (RNNs, LSTMs), transformers process entire 
        sequences in parallel using self-attention mechanisms.
        
        The key innovation is the attention mechanism, which allows the model to weigh the importance 
        of different words in a sequence when processing each word. Multi-head attention enables the 
        model to attend to different aspects of the input simultaneously.
        
        The transformer consists of an encoder and decoder. The encoder processes the input sequence 
        and creates contextual representations. The decoder generates the output sequence, attending 
        to both previously generated tokens and the encoder's output.
        
        Position encodings are added to input embeddings to provide information about token positions, 
        since transformers don't inherently understand sequence order. Feed-forward networks and layer 
        normalization complete the architecture.
        
        Transformer variants include encoder-only models (BERT), decoder-only models (GPT), and 
        encoder-decoder models (T5, BART).""",
        metadata={"source": "transformers.txt", "category": "advanced", "topic": "transformers"}
    ),
    Document(
        page_content="""Large Language Models (LLMs)
        
        Large Language Models are neural networks trained on massive amounts of text data, typically 
        containing billions or trillions of parameters. Examples include GPT-4, Claude, PaLM, and LLaMA.
        
        LLMs exhibit emergent capabilities not present in smaller models: few-shot learning (learning 
        from just a few examples), chain-of-thought reasoning (breaking down complex problems), and 
        cross-task generalization (applying knowledge across different domains).
        
        Training involves two main phases: pretraining on large text corpora using self-supervised 
        objectives (like next-token prediction), and fine-tuning on specific tasks or using techniques 
        like Reinforcement Learning from Human Feedback (RLHF) to align with human preferences.
        
        Challenges include: computational cost (training requires massive GPU clusters), hallucination 
        (generating plausible but incorrect information), bias (reflecting biases in training data), 
        and knowledge cutoff (information only current up to training date).
        
        Applications span code generation, creative writing, analysis, tutoring, and task automation.""",
        metadata={"source": "llms.txt", "category": "advanced", "topic": "llms"}
    ),
    Document(
        page_content="""Prompt Engineering
        
        Prompt engineering is the practice of designing effective inputs (prompts) to elicit desired 
        outputs from language models. Since LLMs are highly sensitive to input phrasing, well-crafted 
        prompts can significantly improve performance.
        
        Key techniques include: Zero-shot prompting (asking the model to perform tasks without examples), 
        Few-shot prompting (providing a few examples before the actual task), Chain-of-thought prompting 
        (encouraging step-by-step reasoning), and Role prompting (asking the model to adopt a specific 
        persona or expertise level).
        
        Effective prompts are clear, specific, and provide sufficient context. Breaking complex tasks 
        into smaller steps, using delimiters to separate different parts of the prompt, and specifying 
        the desired output format all improve results.
        
        Advanced techniques include tree-of-thought (exploring multiple reasoning paths), self-consistency 
        (generating multiple answers and selecting the most common), and retrieval-augmented prompting 
        (including relevant context from external sources).""",
        metadata={"source": "prompt_engineering.txt", "category": "intermediate", "topic": "prompting"}
    )
]

print(f"📄 Created {len(documents)} sample documents")
print(f"📊 Topics covered: {', '.join(set([doc.metadata['topic'] for doc in documents]))}")

📄 Created 8 sample documents
📊 Topics covered: rag, embeddings, transformers, nlp, neural_networks, llms, machine_learning, prompting


### Create Vector Store for Retrieval

We'll split documents into chunks and create a FAISS vector store for all examples.

In [11]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

splits = text_splitter.split_documents(documents)

print(f"📝 Split {len(documents)} documents into {len(splits)} chunks")

# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(splits, embeddings)

print(f"✅ Created FAISS vector store with {vectorstore.index.ntotal} vectors")

# Create base retriever for comparison
base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("✅ Base retriever ready")

📝 Split 8 documents into 27 chunks
✅ Created FAISS vector store with 27 vectors
✅ Base retriever ready


### Test Base Retriever

Let's test the basic retriever to establish a baseline.

In [12]:
# Test query
test_query = "How do transformers work?"

# Retrieve documents
docs = base_retriever.invoke(test_query)

print(f"Query: '{test_query}'\n")
print(f"📚 Retrieved {len(docs)} documents:\n")

for i, doc in enumerate(docs, 1):
    print(f"{i}. Topic: {doc.metadata.get('topic', 'N/A')}")
    print(f"   Content preview: {doc.page_content[:150]}...\n")

Query: 'How do transformers work?'

📚 Retrieved 3 documents:

1. Topic: transformers
   Content preview: The transformer consists of an encoder and decoder. The encoder processes the input sequence 
        and creates contextual representations. The deco...

2. Topic: transformers
   Content preview: Transformer Architecture

        The Transformer architecture, introduced by Vaswani et al. in 2017, has become the foundation 
        of modern NLP...

3. Topic: nlp
   Content preview: The transformer architecture, introduced in the 'Attention is All You Need' paper, revolutionized 
        NLP by using self-attention mechanisms to p...



---

<a id='multiquery'></a>
## 3. MultiQueryRetriever - Multiple Perspectives 🔄

### 🔰 BEGINNER: Understanding the Problem

#### The Query Phrasing Problem

Different ways of asking the same question can retrieve different results:

- "How do transformers work?" ❓
- "Explain the transformer architecture" ❓
- "What is the mechanism behind transformer models?" ❓

These questions have the same intent but different phrasing. A single similarity search might miss relevant documents.

#### How MultiQueryRetriever Solves This

**MultiQueryRetriever** uses an LLM to:
1. Generate multiple variations of the user's query (different perspectives)
2. Retrieve documents for each query variation
3. Combine and deduplicate results
4. Return the most comprehensive set of relevant documents

```
User Query → LLM generates variants → Retrieve for each → Combine → Return unique docs
```

**Benefits:**
- ✅ Better recall (find more relevant documents)
- ✅ Handles query phrasing variations
- ✅ Automatic query expansion
- ✅ More comprehensive results

**Trade-offs:**
- ⚠️ Slower (generates queries + multiple retrievals)
- ⚠️ Higher cost (LLM API calls for query generation)
- ⚠️ May retrieve more documents than needed

---

### 🔰 BEGINNER: Basic MultiQueryRetriever Usage

In [13]:
# Create a MultiQueryRetriever
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Enable logging to see generated queries
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# Create MultiQueryRetriever from base retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

print("✅ MultiQueryRetriever created")

✅ MultiQueryRetriever created


In [14]:
# Test with a query
query = "What are the benefits of RAG?"

print(f"Original query: '{query}'\n")
print("=" * 80)
print("Generated query variations (check logs above):\n")

# Retrieve documents
docs = multi_query_retriever.invoke(query)

print("=" * 80)
print(f"\n📚 Retrieved {len(docs)} unique documents (after deduplication):\n")

for i, doc in enumerate(docs, 1):
    print(f"{i}. Topic: {doc.metadata.get('topic', 'N/A')}")
    print(f"   Preview: {doc.page_content[:120]}...\n")

Original query: 'What are the benefits of RAG?'

Generated query variations (check logs above):


📚 Retrieved 3 unique documents (after deduplication):

1. Topic: rag
   Preview: Benefits of RAG include: reduced hallucination (answers are grounded in retrieved facts), ability 
        to update kno...

2. Topic: rag
   Preview: Retrieval-Augmented Generation (RAG)

        RAG is a powerful technique that combines the strengths of retrieval syste...

3. Topic: rag
   Preview: The RAG process involves three main steps: (1) Query formulation and retrieval - converting user 
        queries into e...



### 🎓 INTERMEDIATE: Custom Query Generation Prompt

In [17]:
# Create a custom prompt for query generation
from langchain_core.prompts import PromptTemplate

# Custom prompt that generates more focused variations
custom_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI assistant helping to improve search queries. 
    Given the original question, generate 4 different versions of it to retrieve relevant documents.
    Each version should approach the question from a different angle or use different terminology.
    
    Original question: {question}
    
    Provide these alternative questions separated by newlines:"""
)

# Create MultiQueryRetriever with custom prompt
from langchain_classic.chains import LLMChain

llm_chain = LLMChain(llm=llm, prompt=custom_prompt)

multi_query_custom = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    prompt=custom_prompt
)

print("✅ MultiQueryRetriever with custom prompt created")

✅ MultiQueryRetriever with custom prompt created


/var/folders/cj/13vbmk7n7fqgmdnqjjwn1bx80000gn/T/ipykernel_30584/1914816818.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=llm, prompt=custom_prompt)


In [18]:
# Test custom prompt
query = "How does deep learning differ from traditional machine learning?"

print(f"Query: '{query}'\n")
print("=" * 80)

docs = multi_query_custom.invoke(query)

print("=" * 80)
print(f"\n📚 Retrieved {len(docs)} documents:\n")

for i, doc in enumerate(docs[:3], 1):  # Show top 3
    print(f"{i}. Topic: {doc.metadata.get('topic', 'N/A')} | Category: {doc.metadata.get('category', 'N/A')}")
    print(f"   {doc.page_content[:150]}...\n")

Query: 'How does deep learning differ from traditional machine learning?'


📚 Retrieved 3 documents:

1. Topic: neural_networks | Category: advanced
   Deep Learning and Neural Networks

        Deep learning is a specialized branch of machine learning based on artificial neural networks. 
        The...

2. Topic: machine_learning | Category: basics
   There are three main types of machine learning: supervised learning (learning from labeled data), 
        unsupervised learning (finding patterns in ...

3. Topic: neural_networks | Category: advanced
   Deep neural networks contain multiple hidden layers between the input and output layers, allowing 
        them to learn hierarchical representations ...



### 🎓 INTERMEDIATE: Comparison - Base vs MultiQuery Retriever

In [19]:
# Compare base retriever vs MultiQueryRetriever
test_query = "What are embeddings used for?"

print(f"Test Query: '{test_query}'\n")
print("=" * 80)

# Base retriever
base_docs = base_retriever.invoke(test_query)
print("\n📊 BASE RETRIEVER:")
print(f"Retrieved {len(base_docs)} documents")
for i, doc in enumerate(base_docs, 1):
    print(f"  {i}. {doc.metadata.get('topic', 'N/A')}")

print("\n" + "=" * 80)

# MultiQuery retriever (suppress logs for cleaner output)
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)
multi_docs = multi_query_retriever.invoke(test_query)

print("\n📊 MULTIQUERY RETRIEVER:")
print(f"Retrieved {len(multi_docs)} unique documents")
for i, doc in enumerate(multi_docs, 1):
    print(f"  {i}. {doc.metadata.get('topic', 'N/A')}")

print("\n" + "=" * 80)
print(f"\n✅ MultiQuery found {len(multi_docs) - len(base_docs)} additional documents")

Test Query: 'What are embeddings used for?'


📊 BASE RETRIEVER:
Retrieved 3 documents
  1. embeddings
  2. embeddings
  3. transformers


📊 MULTIQUERY RETRIEVER:
Retrieved 4 unique documents
  1. embeddings
  2. embeddings
  3. embeddings
  4. transformers


✅ MultiQuery found 1 additional documents


### 💡 MultiQueryRetriever Best Practices

#### When to Use:
- ✅ User queries are diverse and unpredictable
- ✅ You need comprehensive coverage of a topic
- ✅ Query quality matters more than speed
- ✅ Your knowledge base has diverse terminology

#### When NOT to Use:
- ❌ Low-latency requirements (real-time chat)
- ❌ Cost-sensitive applications (extra LLM calls)
- ❌ Simple, well-defined queries
- ❌ Small document collections (basic retrieval works fine)

#### Tips:
1. **Tune the number of generated queries**: Default is 3-5, adjust based on your needs
2. **Cache query variations**: For common queries, cache the generated variations
3. **Monitor costs**: Each retrieval requires an LLM API call for query generation
4. **Customize prompts**: Tailor query generation to your domain and use case
5. **Combine with other techniques**: Works well with contextual compression

---

<a id='compression'></a>
## 4. Contextual Compression - Token Efficiency 🗜️

### 🔰 BEGINNER: Understanding the Problem

#### The Irrelevant Content Problem

Retrieved documents often contain:
- ❌ Irrelevant sections
- ❌ Unnecessary background information
- ❌ Redundant content

**Result**: Wasted tokens → Higher costs → Slower responses → Context window pollution

#### How Contextual Compression Solves This

**Contextual Compression** extracts only the relevant portions of retrieved documents:

```
Retrieved Docs → Compressor extracts relevant parts → Compressed Docs → Send to LLM
```

**Types of Compressors:**

1. **LLMChainExtractor**: Uses an LLM to extract relevant sentences
2. **EmbeddingsFilter**: Filters chunks based on embedding similarity
3. **DocumentCompressorPipeline**: Chains multiple compressors

**Benefits:**
- ✅ Reduced token usage (lower costs)
- ✅ Faster LLM responses (less content to process)
- ✅ Better context utilization (only relevant info)
- ✅ Improved answer quality (less noise)

**Trade-offs:**
- ⚠️ Compression takes time (adds latency)
- ⚠️ LLM-based compression costs tokens
- ⚠️ Risk of over-compression (losing important context)

---

### 🔰 BEGINNER: LLMChainExtractor Compressor

In [ ]:
# Create LLMChainExtractor compressor
# This uses an LLM to extract only relevant sentences from documents
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
compressor = LLMChainExtractor.from_llm(llm)

# Create ContextualCompressionRetriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("✅ LLMChainExtractor compression retriever created")

✅ LLMChainExtractor compression retriever created


In [27]:
# Test compression retriever
query = "What is the transformer architecture?"

print(f"Query: '{query}'\n")
print("=" * 80)

# Get compressed documents
compressed_docs = compression_retriever.invoke(query)

print(f"\n📚 Retrieved and compressed {len(compressed_docs)} documents:\n")

for i, doc in enumerate(compressed_docs, 1):
    print(f"{i}. Topic: {doc.metadata.get('topic', 'N/A')}")
    print(f"   Compressed content:\n   {doc.page_content}")
    print()

Query: 'What is the transformer architecture?'


📚 Retrieved and compressed 3 documents:

1. Topic: transformers
   Compressed content:
   The Transformer architecture, introduced by Vaswani et al. in 2017, has become the foundation 
of modern NLP. Unlike previous sequential models (RNNs, LSTMs), transformers process entire 
sequences in parallel using self-attention mechanisms.

2. Topic: transformers
   Compressed content:
   The transformer consists of an encoder and decoder. The encoder processes the input sequence and creates contextual representations. The decoder generates the output sequence, attending to both previously generated tokens and the encoder's output.

3. Topic: nlp
   Compressed content:
   The transformer architecture, introduced in the 'Attention is All You Need' paper, revolutionized 
        NLP by using self-attention mechanisms to process sequential data in parallel.



### 🔰 BEGINNER: Comparison - Before vs After Compression

In [28]:
# Compare token usage: base retriever vs compression retriever
import tiktoken

def count_tokens(text: str, model: str = "gpt-3.5-turbo") -> int:
    """Count tokens in text using tiktoken."""
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

query = "How does RAG improve language model performance?"

print(f"Query: '{query}'\n")
print("=" * 80)

# Base retriever
base_docs = base_retriever.invoke(query)
base_content = "\n\n".join([doc.page_content for doc in base_docs])
base_tokens = count_tokens(base_content)

print("\n📊 BASE RETRIEVER (No Compression):")
print(f"  Documents: {len(base_docs)}")
print(f"  Total tokens: {base_tokens}")
print(f"  Content length: {len(base_content)} characters")

# Compression retriever
compressed_docs = compression_retriever.invoke(query)
compressed_content = "\n\n".join([doc.page_content for doc in compressed_docs])
compressed_tokens = count_tokens(compressed_content)

print("\n📊 COMPRESSION RETRIEVER:")
print(f"  Documents: {len(compressed_docs)}")
print(f"  Total tokens: {compressed_tokens}")
print(f"  Content length: {len(compressed_content)} characters")

# Calculate savings
token_savings = base_tokens - compressed_tokens
savings_percentage = (token_savings / base_tokens) * 100 if base_tokens > 0 else 0

print("\n" + "=" * 80)
print(f"\n💰 SAVINGS:")
print(f"  Token reduction: {token_savings} tokens ({savings_percentage:.1f}%)")
print(f"  Character reduction: {len(base_content) - len(compressed_content)} characters")

# Estimate cost savings (approximate)
cost_per_1k_tokens = 0.0015  # GPT-3.5-turbo input cost
cost_savings = (token_savings / 1000) * cost_per_1k_tokens
print(f"  Estimated cost savings per query: ${cost_savings:.6f}")

Query: 'How does RAG improve language model performance?'


📊 BASE RETRIEVER (No Compression):
  Documents: 3
  Total tokens: 225
  Content length: 1202 characters

📊 COMPRESSION RETRIEVER:
  Documents: 3
  Total tokens: 146
  Content length: 812 characters


💰 SAVINGS:
  Token reduction: 79 tokens (35.1%)
  Character reduction: 390 characters
  Estimated cost savings per query: $0.000119


### 🎓 INTERMEDIATE: EmbeddingsFilter Compressor

**EmbeddingsFilter** filters document chunks based on embedding similarity to the query. It's faster and cheaper than LLM-based extraction.

In [30]:
# Create EmbeddingsFilter compressor
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter
from langchain_text_splitters import CharacterTextSplitter

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Split documents into smaller chunks for fine-grained filtering
splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separator=". "
)

# Create filter with similarity threshold
embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.7  # Only keep chunks with similarity > 0.7
)

# Create compression retriever with embeddings filter
embedding_compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever
)

print("✅ EmbeddingsFilter compression retriever created")

✅ EmbeddingsFilter compression retriever created


In [31]:
# Test embeddings filter
query = "What are the main components of a neural network?"

print(f"Query: '{query}'\n")
print("=" * 80)

# Retrieve with embeddings filter
filtered_docs = embedding_compression_retriever.invoke(query)

print(f"\n📚 Retrieved {len(filtered_docs)} relevant chunks (similarity > 0.7):\n")

for i, doc in enumerate(filtered_docs, 1):
    print(f"{i}. {doc.page_content}")
    print()

Query: 'What are the main components of a neural network?'


📚 Retrieved 0 relevant chunks (similarity > 0.7):



### 🎓 INTERMEDIATE: DocumentCompressorPipeline - Chaining Compressors

Combine multiple compressors for maximum efficiency!

In [32]:
# Create a compression pipeline
# Step 1: Split into smaller chunks
# Step 2: Filter by embedding similarity
# Step 3: Extract relevant sentences with LLM

from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter


# Create pipeline components
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator=". ")
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.65)
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
llm_extractor = LLMChainExtractor.from_llm(llm)

# Create pipeline
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[
        splitter,           # Split into chunks
        embeddings_filter,  # Filter by similarity
        redundant_filter,   # Remove redundant chunks
        llm_extractor       # Extract relevant sentences
    ]
)

# Create compression retriever with pipeline
pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever
)

print("✅ Multi-stage compression pipeline created")

✅ Multi-stage compression pipeline created


In [33]:
# Test pipeline compression
query = "Explain how attention mechanisms work"

print(f"Query: '{query}'\n")
print("=" * 80)
print("Processing through pipeline:")
print("  1. Splitting documents into chunks")
print("  2. Filtering by embedding similarity")
print("  3. Removing redundant content")
print("  4. Extracting relevant sentences")
print("=" * 80)

# Retrieve with pipeline
pipeline_docs = pipeline_retriever.invoke(query)

print(f"\n📚 Final output: {len(pipeline_docs)} highly relevant excerpts:\n")

for i, doc in enumerate(pipeline_docs, 1):
    print(f"{i}. {doc.page_content}")
    print()

# Calculate compression
pipeline_content = "\n".join([doc.page_content for doc in pipeline_docs])
pipeline_tokens = count_tokens(pipeline_content)

print("=" * 80)
print(f"\n📊 Pipeline Compression Stats:")
print(f"  Final tokens: {pipeline_tokens}")
print(f"  Content length: {len(pipeline_content)} characters")

Query: 'Explain how attention mechanisms work'

Processing through pipeline:
  1. Splitting documents into chunks
  2. Filtering by embedding similarity
  3. Removing redundant content
  4. Extracting relevant sentences

📚 Final output: 0 highly relevant excerpts:


📊 Pipeline Compression Stats:
  Final tokens: 0
  Content length: 0 characters


### 💡 Contextual Compression Best Practices

#### Choosing the Right Compressor:

| Compressor | Speed | Cost | Quality | Use Case |
|------------|-------|------|---------|----------|
| **EmbeddingsFilter** | Fast ⚡ | Low 💰 | Good ✅ | Real-time apps, high throughput |
| **LLMChainExtractor** | Slow 🐌 | High 💸 | Excellent ⭐ | Quality-critical, low volume |
| **Pipeline** | Medium ⏱️ | Medium 💵 | Excellent ⭐ | Best balance |

#### When to Use:
- ✅ Retrieved documents are long (>500 tokens)
- ✅ Token costs are significant
- ✅ Context window is limited
- ✅ Documents contain mixed relevant/irrelevant content

#### When NOT to Use:
- ❌ Documents are already concise
- ❌ Need extremely low latency
- ❌ All retrieved content is relevant

#### Tips:
1. **Tune similarity threshold**: Start with 0.7, adjust based on results
2. **Balance pipeline stages**: Don't over-compress
3. **Monitor compression ratio**: Track tokens saved vs. quality
4. **Cache compressed results**: For common queries
5. **Test on your data**: Different domains need different settings

---

<a id='custom'></a>
## 5. Custom Retrievers - Full Control 🔧

### 🔰 BEGINNER: When to Build Custom Retrievers

#### Scenarios Requiring Custom Retrievers:

- 🎯 **Business-specific logic**: Filter by user permissions, departments, dates
- 🔀 **Hybrid retrieval**: Combine keyword search + semantic search
- 🎲 **Custom scoring**: Re-rank based on custom relevance metrics
- 📊 **Metadata filtering**: Complex queries on document metadata
- ⚡ **Performance optimization**: Custom caching, batching strategies

#### Two Approaches:

1. **@chain Decorator** (Simple, for basic custom logic)
2. **BaseRetriever Class** (Advanced, for production-grade retrievers)

---

### 🔰 BEGINNER: Method 1 - Using @chain Decorator

Quick way to create custom retrieval logic (reviewed from Notebook 06).

In [ ]:
# Simple custom retriever using @chain decorator
from langchain_core.runnables import chain

@chain
def custom_keyword_retriever(query: str) -> List[Document]:
    """
    Custom retriever that filters documents by keyword presence.
    Combines semantic search with keyword filtering.
    """
    # Get semantic search results
    docs = base_retriever.invoke(query)
    
    # Define keywords to filter by
    required_keywords = ["transformer", "attention", "neural"]
    
    # Filter documents containing at least one keyword
    filtered_docs = [
        doc for doc in docs
        if any(keyword.lower() in doc.page_content.lower() for keyword in required_keywords)
    ]
    
    return filtered_docs if filtered_docs else docs[:1]  # Return at least one doc

print("✅ Custom keyword retriever created")

In [ ]:
# Test custom keyword retriever
query = "How do modern AI models process language?"

print(f"Query: '{query}'\n")
print("=" * 80)

docs = custom_keyword_retriever.invoke(query)

print(f"\n📚 Retrieved {len(docs)} documents (filtered by keywords: transformer, attention, neural):\n")

for i, doc in enumerate(docs, 1):
    print(f"{i}. Topic: {doc.metadata.get('topic', 'N/A')}")
    print(f"   {doc.page_content[:200]}...\n")

### 🎓 INTERMEDIATE: Method 2 - Inheriting from BaseRetriever

**Production-grade approach** with validation, error handling, and full control.

In [ ]:
# Advanced: Create a custom retriever by inheriting from BaseRetriever
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from pydantic import Field
from typing import List

class MetadataFilterRetriever(BaseRetriever):
    """
    Custom retriever that filters documents by metadata attributes.
    
    Example: Filter by category, topic, date, etc.
    """
    
    vectorstore: FAISS = Field(description="Vector store to search")
    filter_key: str = Field(description="Metadata key to filter by")
    filter_value: str = Field(description="Metadata value to filter for")
    k: int = Field(default=3, description="Number of documents to return")
    
    class Config:
        arbitrary_types_allowed = True
    
    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        """
        Retrieve documents matching the query and metadata filter.
        """
        # Perform similarity search with higher k to allow filtering
        docs = self.vectorstore.similarity_search(query, k=self.k * 3)
        
        # Filter by metadata
        filtered_docs = [
            doc for doc in docs
            if doc.metadata.get(self.filter_key) == self.filter_value
        ]
        
        # Return top k filtered documents
        return filtered_docs[:self.k]

# Create instances for different categories
advanced_retriever = MetadataFilterRetriever(
    vectorstore=vectorstore,
    filter_key="category",
    filter_value="advanced",
    k=3
)

basics_retriever = MetadataFilterRetriever(
    vectorstore=vectorstore,
    filter_key="category",
    filter_value="basics",
    k=3
)

print("✅ MetadataFilterRetriever created")

In [ ]:
# Test metadata filter retriever
query = "How does machine learning work?"

print(f"Query: '{query}'\n")
print("=" * 80)

# Retrieve from "basics" category
basic_docs = basics_retriever.invoke(query)
print("\n📚 BASICS CATEGORY:")
for i, doc in enumerate(basic_docs, 1):
    print(f"  {i}. {doc.metadata.get('topic', 'N/A')} (Category: {doc.metadata.get('category', 'N/A')})")

print("\n" + "=" * 80)

# Retrieve from "advanced" category
adv_docs = advanced_retriever.invoke(query)
print("\n📚 ADVANCED CATEGORY:")
for i, doc in enumerate(adv_docs, 1):
    print(f"  {i}. {doc.metadata.get('topic', 'N/A')} (Category: {doc.metadata.get('category', 'N/A')})")

### 🎓 INTERMEDIATE: Hybrid Retriever (Keyword + Semantic)

Combine BM25 keyword search with semantic search for best of both worlds.

In [ ]:
# Create a hybrid retriever combining keyword and semantic search
from typing import List, Dict
import re

class HybridRetriever(BaseRetriever):
    """
    Hybrid retriever combining keyword matching and semantic search.
    
    Scores documents using:
    - Keyword matching (BM25-like)
    - Semantic similarity (embeddings)
    """
    
    vectorstore: FAISS = Field(description="Vector store for semantic search")
    all_docs: List[Document] = Field(description="All documents for keyword search")
    k: int = Field(default=3, description="Number of documents to return")
    semantic_weight: float = Field(default=0.7, description="Weight for semantic score (0-1)")
    
    class Config:
        arbitrary_types_allowed = True
    
    def _keyword_search(self, query: str, docs: List[Document]) -> Dict[int, float]:
        """Simple keyword scoring based on term frequency."""
        query_terms = set(query.lower().split())
        scores = {}
        
        for i, doc in enumerate(docs):
            content_lower = doc.page_content.lower()
            # Count matching terms
            matches = sum(1 for term in query_terms if term in content_lower)
            scores[i] = matches / len(query_terms) if query_terms else 0
        
        return scores
    
    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        """
        Retrieve documents using hybrid scoring.
        """
        # Semantic search with scores
        semantic_results = self.vectorstore.similarity_search_with_score(query, k=self.k * 2)
        
        # Normalize semantic scores (FAISS returns distances, lower is better)
        max_distance = max([score for _, score in semantic_results]) if semantic_results else 1
        semantic_scores = {
            i: 1 - (score / max_distance) for i, (_, score) in enumerate(semantic_results)
        }
        
        # Keyword search
        retrieved_docs = [doc for doc, _ in semantic_results]
        keyword_scores = self._keyword_search(query, retrieved_docs)
        
        # Combine scores
        hybrid_scores = {}
        for i in range(len(retrieved_docs)):
            semantic = semantic_scores.get(i, 0)
            keyword = keyword_scores.get(i, 0)
            hybrid_scores[i] = (
                self.semantic_weight * semantic +
                (1 - self.semantic_weight) * keyword
            )
        
        # Sort by hybrid score
        sorted_indices = sorted(hybrid_scores.keys(), key=lambda i: hybrid_scores[i], reverse=True)
        
        # Return top k documents
        return [retrieved_docs[i] for i in sorted_indices[:self.k]]

# Create hybrid retriever
hybrid_retriever = HybridRetriever(
    vectorstore=vectorstore,
    all_docs=splits,
    k=3,
    semantic_weight=0.7  # 70% semantic, 30% keyword
)

print("✅ HybridRetriever created (70% semantic + 30% keyword)")

In [ ]:
# Test hybrid retriever
query = "transformer attention mechanism"

print(f"Query: '{query}'\n")
print("=" * 80)

# Hybrid retrieval
hybrid_docs = hybrid_retriever.invoke(query)

print(f"\n📚 Hybrid Retrieval (Semantic + Keyword):\n")
for i, doc in enumerate(hybrid_docs, 1):
    print(f"{i}. Topic: {doc.metadata.get('topic', 'N/A')}")
    print(f"   {doc.page_content[:180]}...\n")

### 🚀 ADVANCED: LLM-Based Re-Ranking Retriever

Use an LLM to re-rank retrieved documents based on relevance to the query.

In [ ]:
# Create a re-ranking retriever using LLM
from langchain_core.prompts import ChatPromptTemplate

class LLMReRankingRetriever(BaseRetriever):
    """
    Retriever that uses an LLM to re-rank documents based on relevance.
    """
    
    base_retriever: BaseRetriever = Field(description="Base retriever to get initial documents")
    llm: ChatOpenAI = Field(description="LLM for re-ranking")
    k: int = Field(default=3, description="Number of documents to return after re-ranking")
    
    class Config:
        arbitrary_types_allowed = True
    
    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        """
        Retrieve and re-rank documents using LLM.
        """
        # Get initial documents (retrieve more than needed)
        initial_docs = self.base_retriever.invoke(query)
        
        if len(initial_docs) <= self.k:
            return initial_docs
        
        # Create re-ranking prompt
        rerank_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a relevance scoring assistant."),
            ("user", """Given the query and document, rate the relevance on a scale of 0-10.
            Only respond with a number.
            
            Query: {query}
            
            Document: {document}
            
            Relevance score (0-10):""")
        ])
        
        # Score each document
        scored_docs = []
        for doc in initial_docs:
            try:
                score_response = self.llm.invoke(
                    rerank_prompt.format_messages(
                        query=query,
                        document=doc.page_content[:500]  # Limit content length
                    )
                )
                score = float(score_response.content.strip())
                scored_docs.append((doc, score))
            except:
                scored_docs.append((doc, 0))  # Default score if parsing fails
        
        # Sort by score and return top k
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scored_docs[:self.k]]

# Create re-ranking retriever
rerank_retriever = LLMReRankingRetriever(
    base_retriever=base_retriever,
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    k=3
)

print("✅ LLM Re-Ranking Retriever created")

In [ ]:
# Test re-ranking retriever
query = "What are the key innovations in transformer models?"

print(f"Query: '{query}'\n")
print("=" * 80)
print("Re-ranking documents using LLM...\n")

reranked_docs = rerank_retriever.invoke(query)

print("=" * 80)
print(f"\n📚 Re-Ranked Documents (Top {len(reranked_docs)}):\n")

for i, doc in enumerate(reranked_docs, 1):
    print(f"{i}. Topic: {doc.metadata.get('topic', 'N/A')}")
    print(f"   {doc.page_content[:200]}...\n")

### 💡 Custom Retriever Best Practices

#### When to Use Each Method:

| Method | Complexity | Use Case | Best For |
|--------|-----------|----------|----------|
| **@chain decorator** | Low | Simple transformations | Quick prototypes, basic logic |
| **BaseRetriever** | High | Production systems | Full control, validation, async support |

#### BaseRetriever Implementation Checklist:

✅ **Required:**
- Inherit from `BaseRetriever`
- Implement `_get_relevant_documents()` method
- Define fields with Pydantic `Field`
- Set `Config.arbitrary_types_allowed = True` (for non-Pydantic types)

✅ **Recommended:**
- Add input validation
- Handle errors gracefully
- Add logging for debugging
- Support async with `_aget_relevant_documents()` (optional)
- Add docstrings

#### Tips:

1. **Start simple**: Begin with @chain, upgrade to BaseRetriever when needed
2. **Test thoroughly**: Custom retrievers are harder to debug
3. **Cache results**: Especially for expensive operations (LLM re-ranking)
4. **Monitor performance**: Track retrieval time and quality
5. **Document behavior**: Custom logic should be well-documented

---

<a id='combinations'></a>
## 6. Advanced Combinations 🔀

Combine multiple techniques for optimal results!

### Combination 1: MultiQuery + Contextual Compression

Generate multiple query variations AND compress results for best quality with token efficiency.

In [ ]:
# Combine MultiQueryRetriever with Contextual Compression

# Step 1: Create MultiQueryRetriever
multi_query_base = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

# Step 2: Wrap with ContextualCompressionRetriever
multi_query_compressed = ContextualCompressionRetriever(
    base_compressor=LLMChainExtractor.from_llm(llm),
    base_retriever=multi_query_base
)

print("✅ MultiQuery + Compression retriever created")

In [ ]:
# Test combined retriever
query = "How do LLMs handle context?"

print(f"Query: '{query}'\n")
print("=" * 80)
print("Step 1: Generating multiple query variations...")
print("Step 2: Retrieving documents for each variation...")
print("Step 3: Compressing retrieved documents...")
print("=" * 80)

# Suppress logs for cleaner output
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)

combined_docs = multi_query_compressed.invoke(query)

print(f"\n📚 Final Result: {len(combined_docs)} compressed, highly relevant documents:\n")

for i, doc in enumerate(combined_docs, 1):
    print(f"{i}. {doc.page_content}\n")

# Calculate token efficiency
combined_content = "\n".join([doc.page_content for doc in combined_docs])
combined_tokens = count_tokens(combined_content)
print(f"\n💰 Total tokens: {combined_tokens}")

### Combination 2: Custom Retriever + Compression Pipeline

Use custom retrieval logic (metadata filtering) followed by compression.

In [ ]:
# Combine custom metadata filtering with compression

# Create custom metadata retriever (advanced topics only)
custom_advanced = MetadataFilterRetriever(
    vectorstore=vectorstore,
    filter_key="category",
    filter_value="advanced",
    k=5  # Retrieve more before compression
)

# Wrap with compression
custom_compressed = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,  # Use our multi-stage pipeline
    base_retriever=custom_advanced
)

print("✅ Custom Metadata Filtering + Compression pipeline created")

In [ ]:
# Test custom + compression
query = "Explain neural network architectures"

print(f"Query: '{query}'\n")
print("=" * 80)
print("Retrieving from 'advanced' category only + applying compression...")
print("=" * 80)

filtered_compressed_docs = custom_compressed.invoke(query)

print(f"\n📚 Advanced-only, compressed documents: {len(filtered_compressed_docs)}\n")

for i, doc in enumerate(filtered_compressed_docs, 1):
    print(f"{i}. Category: {doc.metadata.get('category', 'N/A')}")
    print(f"   Content: {doc.page_content}\n")

### Combination 3: Complete Advanced RAG Chain

Build a production-ready RAG system using multiple advanced techniques.

In [ ]:
# Build a complete advanced RAG chain
from langchain_core.output_parsers import StrOutputParser

# Components:
# 1. MultiQueryRetriever (better recall)
# 2. Contextual Compression (token efficiency)
# 3. Custom prompt with source attribution

# Retriever (already created above)
advanced_retriever = multi_query_compressed

# LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Advanced prompt with source attribution
advanced_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Answer questions based on the provided context. Always cite which parts of the context you used."),
    ("user", """Context:
{context}

Question: {question}

Provide a comprehensive answer with source attribution:""")
])

# Format documents helper
def format_docs_with_sources(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(f"[Source {i}] {doc.page_content}")
    return "\n\n".join(formatted)

# Build the advanced RAG chain
advanced_rag_chain = (
    {"context": advanced_retriever | format_docs_with_sources, "question": RunnablePassthrough()}
    | advanced_prompt
    | llm
    | StrOutputParser()
)

print("✅ Advanced RAG chain created")
print("   - MultiQuery for better recall")
print("   - Contextual Compression for efficiency")
print("   - Source attribution for transparency")

In [ ]:
# Test advanced RAG chain
question = "What makes transformer models effective for NLP tasks?"

print(f"Question: {question}\n")
print("=" * 80)
print("Processing with advanced RAG pipeline...")
print("=" * 80 + "\n")

answer = advanced_rag_chain.invoke(question)

print("Answer:")
print(answer)

---

<a id='benchmarking'></a>
## 7. Comparison & Benchmarking 📊

Let's compare all the retrieval techniques we've learned!

In [ ]:
# Benchmark different retrieval techniques
import time

test_query = "How do neural networks learn from data?"

print(f"Benchmark Query: '{test_query}'\n")
print("=" * 80)

# Suppress logs
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)

results = {}

# 1. Base Retriever
start = time.time()
base_docs = base_retriever.invoke(test_query)
base_time = time.time() - start
base_tokens = count_tokens("\n".join([d.page_content for d in base_docs]))
results['Base'] = {'docs': len(base_docs), 'tokens': base_tokens, 'time': base_time}

# 2. MultiQuery Retriever
start = time.time()
multi_docs = multi_query_retriever.invoke(test_query)
multi_time = time.time() - start
multi_tokens = count_tokens("\n".join([d.page_content for d in multi_docs]))
results['MultiQuery'] = {'docs': len(multi_docs), 'tokens': multi_tokens, 'time': multi_time}

# 3. Contextual Compression
start = time.time()
comp_docs = compression_retriever.invoke(test_query)
comp_time = time.time() - start
comp_tokens = count_tokens("\n".join([d.page_content for d in comp_docs]))
results['Compression'] = {'docs': len(comp_docs), 'tokens': comp_tokens, 'time': comp_time}

# 4. Custom Hybrid
start = time.time()
hybrid_docs = hybrid_retriever.invoke(test_query)
hybrid_time = time.time() - start
hybrid_tokens = count_tokens("\n".join([d.page_content for d in hybrid_docs]))
results['Hybrid'] = {'docs': len(hybrid_docs), 'tokens': hybrid_tokens, 'time': hybrid_time}

# 5. MultiQuery + Compression
start = time.time()
advanced_docs = multi_query_compressed.invoke(test_query)
advanced_time = time.time() - start
advanced_tokens = count_tokens("\n".join([d.page_content for d in advanced_docs]))
results['MultiQuery+Compression'] = {'docs': len(advanced_docs), 'tokens': advanced_tokens, 'time': advanced_time}

print("\n📊 BENCHMARK RESULTS:\n")
print(f"{'Technique':<25} {'Docs':<8} {'Tokens':<10} {'Time (s)':<12} {'Token Efficiency'}")
print("=" * 80)

base_token_count = results['Base']['tokens']
for name, data in results.items():
    efficiency = f"{((base_token_count - data['tokens']) / base_token_count * 100):.1f}% saved" if base_token_count > 0 else "baseline"
    print(f"{name:<25} {data['docs']:<8} {data['tokens']:<10} {data['time']:<12.3f} {efficiency}")

print("=" * 80)

### Comparison Summary Table

| Technique | Recall | Precision | Token Efficiency | Speed | Cost | Use Case |
|-----------|--------|-----------|-----------------|-------|------|----------|
| **Base Retriever** | Medium | Medium | Low | Fast ⚡ | Low 💰 | Simple queries, testing |
| **MultiQueryRetriever** | High ⭐ | Medium | Low | Slow 🐌 | High 💸 | Comprehensive answers |
| **Contextual Compression** | Medium | High ⭐ | High ⭐ | Medium ⏱️ | Medium 💵 | Long documents, cost optimization |
| **Custom Hybrid** | High ⭐ | High ⭐ | Medium | Medium ⏱️ | Low 💰 | Specialized domains |
| **MultiQuery + Compression** | High ⭐ | High ⭐ | High ⭐ | Slow 🐌 | High 💸 | Production quality systems |

---

<a id='best-practices'></a>
## 8. Best Practices & Production Tips 💡

### Production Checklist

#### 1. **Error Handling**

```python
try:
    docs = retriever.invoke(query)
    if not docs:
        # Fallback to base retriever
        docs = base_retriever.invoke(query)
except Exception as e:
    logger.error(f"Retrieval failed: {e}")
    # Return cached results or empty list
    docs = []
```

#### 2. **Caching**

```python
from functools import lru_cache

@lru_cache(maxsize=100)
def cached_retrieve(query: str):
    return retriever.invoke(query)
```

#### 3. **Monitoring**

Track:
- Retrieval latency
- Token usage
- Cost per query
- Cache hit rate
- Quality metrics (relevance scores)

#### 4. **Cost Optimization**

- Use EmbeddingsFilter instead of LLMChainExtractor when possible
- Cache query variations for MultiQueryRetriever
- Batch similar queries
- Set appropriate `k` values (don't over-retrieve)

#### 5. **Testing**

```python
# Create test suite
test_cases = [
    {"query": "...", "expected_topics": [...]},
    # ... more test cases
]

for test in test_cases:
    docs = retriever.invoke(test['query'])
    actual_topics = [d.metadata['topic'] for d in docs]
    assert any(topic in actual_topics for topic in test['expected_topics'])
```

---

### Decision Tree: Which Technique to Use?

```
START
  |
  ├─ Need low latency? ──YES→ Base Retriever or Custom Hybrid
  |
  ├─ Documents are long? ──YES→ Contextual Compression
  |
  ├─ Query quality varies? ──YES→ MultiQueryRetriever
  |
  ├─ Custom business logic? ──YES→ Custom Retriever (BaseRetriever)
  |
  └─ Need best quality? ──YES→ MultiQuery + Compression
```

---

<a id='summary'></a>
## 9. Summary & Exercises 📝

### 🎯 What You Learned

**1. MultiQueryRetriever** 🔄
- Generates multiple query variations using LLM
- Improves recall and handles query phrasing variations
- Trade-off: Slower and more expensive, but better quality

**2. Contextual Compression** 🗜️
- Extracts only relevant portions from retrieved documents
- Three types: LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline
- Reduces tokens by 40-70% while maintaining relevance
- Improves cost efficiency and answer quality

**3. Custom Retrievers** 🔧
- Two methods: @chain decorator (simple) and BaseRetriever class (advanced)
- Full control over retrieval logic
- Examples: Metadata filtering, hybrid search, LLM re-ranking
- Production-ready with validation and error handling

**4. Advanced Combinations** 🔀
- MultiQuery + Compression for best quality and efficiency
- Custom retrievers + Compression for specialized use cases
- Complete production RAG systems

**5. Benchmarking & Best Practices** 📊
- Performance comparison across techniques
- Token efficiency analysis
- Production considerations (caching, monitoring, cost optimization)

---

### 💪 Practice Exercises

#### Exercise 1: Query Expansion Analysis (🔰 Beginner)
**Task**: Create a MultiQueryRetriever and analyze the generated query variations.
- Enable logging to see generated queries
- Test with 3 different queries
- Compare results with base retriever
- Calculate the improvement in document diversity

#### Exercise 2: Cost Optimization with Compression (🔰 Beginner)
**Task**: Implement contextual compression to reduce costs.
- Create an EmbeddingsFilter with threshold 0.75
- Test on long documents from your domain
- Calculate token savings percentage
- Estimate monthly cost savings for 10,000 queries

#### Exercise 3: Time-Aware Custom Retriever (🎓 Intermediate)
**Task**: Build a custom retriever that prioritizes recent documents.
- Inherit from BaseRetriever
- Add a "recency_weight" parameter (0-1)
- Score documents based on date + semantic similarity
- Test with documents having different dates

#### Exercise 4: Multi-Stage Retrieval Pipeline (🎓 Intermediate)
**Task**: Build a pipeline combining all three techniques.
- Stage 1: MultiQueryRetriever (generate variations)
- Stage 2: Custom metadata filter (filter by category)
- Stage 3: Contextual compression (extract relevant portions)
- Compare with base retriever on quality and efficiency

#### Exercise 5: Production RAG System (🚀 Advanced)
**Task**: Build a complete production-ready RAG system.
- Implement advanced retrieval (MultiQuery + Compression)
- Add error handling and fallback mechanisms
- Implement caching for common queries
- Add monitoring (track latency, tokens, cost)
- Create a test suite with 10+ test cases
- Document all components and design decisions

---

### 🔗 Next Steps

- **Notebook 10**: Evaluation & Testing RAG Systems
- **Notebook 11**: Production Deployment & Monitoring
- **LangChain Retrievers**: https://python.langchain.com/docs/modules/data_connection/retrievers/

---

### 📚 Additional Resources

- **MultiQueryRetriever**: https://python.langchain.com/docs/modules/data_connection/retrievers/MultiQueryRetriever
- **Contextual Compression**: https://python.langchain.com/docs/modules/data_connection/retrievers/contextual_compression
- **Custom Retrievers**: https://python.langchain.com/docs/modules/data_connection/retrievers/custom_retriever
- **RAG Paper**: https://arxiv.org/abs/2005.11401

---

**Congratulations!** 🎉 You've mastered advanced retrieval techniques!

You can now:
- 🔄 Handle diverse query phrasings with MultiQueryRetriever
- 🗜️ Optimize costs with Contextual Compression
- 🔧 Build custom retrievers for specialized use cases
- 🚀 Combine techniques for production-grade RAG systems
- 📊 Benchmark and optimize retrieval performance

Keep building amazing AI applications! 🚀